# Step 2: Data Generation

Generate synthetic EMR data for training.

## Risk Level Calculation

| Score | Risk Level |
|-------|------------|
| 0-2 | LOW |
| 3-5 | MEDIUM |
| 6-8 | HIGH |
| 9+ | CRITICAL |

## Prerequisites

- Run **01_setup_infrastructure.ipynb** first

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## Generate Synthetic Patient Data

The `HistoricalDataGenerator` creates realistic synthetic patient records with correlated clinical features. It performs the following steps:

1. **Generate Raw Data** (`RAW_PATIENT_DATA`) - Creates patient records with:
   - Demographics (age, gender, BMI)
   - Vital signs (heart rate, blood pressure, temperature, respiratory rate, oxygen saturation)
   - Lab values (glucose, creatinine, hemoglobin, WBC count)
   - Clinical context (diagnosis, comorbidities, admission type, insurance)
   - Risk level labels (LOW, MEDIUM, HIGH, CRITICAL) calculated from clinical indicators

2. **Create Baseline Sample** (`BASELINE_PATIENT_DATA`) - A 20% random sample used for:
   - Data drift detection
   - Monitoring distribution shifts over time

3. **Create Test Split** (`TEST_PATIENT_DATA`) - A 20% holdout set used for:
   - Model evaluation after training
   - Promotion criteria validation

**Expected Risk Distribution**: ~40% LOW, ~35% MEDIUM, ~18% HIGH, ~7% CRITICAL

In [ ]:
%autoreload
from data.historical import HistoricalDataGenerator

generator = HistoricalDataGenerator(
    session=session,
    database=config.snowflake.database,
    schema_name=config.snowflake.schema_name,
)

NUM_RECORDS = 10000

data_result = generator.run(
    num_records=NUM_RECORDS,
    table_name=config.tables.raw_data,
    create_baseline=True,
    create_test_split=False,
)

print(f"\nData generation complete:")
print(f"  Main table: {data_result['main_table']}")
print(f"  Baseline table: {data_result['baseline_table']}")

## Explore Generated Data

In [ ]:
df = session.table(config.full_raw_table).to_pandas()

print(f"Dataset shape: {df.shape}")
print(f"\nRisk Level Distribution:")
print(df["RISK_LEVEL"].value_counts(normalize=True).round(3))

print(f"\nSample Records:")
df[["PATIENT_ID", "AGE", "GENDER", "HEART_RATE", "SYSTOLIC_BP", "OXYGEN_SATURATION", 
    "GLUCOSE_LEVEL", "CREATININE", "COMORBIDITY_COUNT", "INSURANCE_TYPE", "RISK_LEVEL"]].head(10)

## Next Step

Continue to **03_preprocessing.ipynb**